__Task_1__ - import data and libraries, fillna

In [59]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from category_encoders import HashingEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

In [60]:
df = pd.read_csv("data/training.csv")
print(df.shape)
df.head(2)

(72983, 34)


,RefId,IsBadBuy,PurchDate,Auction,VehYear,VehicleAge,Make,Model,Trim,SubModel,...,MMRCurrentRetailAveragePrice,MMRCurrentRetailCleanPrice,PRIMEUNIT,AUCGUART,BYRNO,VNZIP1,VNST,VehBCost,IsOnlineSale,WarrantyCost
0,1,0,12/7/2009,ADESA,2006,3,MAZDA,MAZDA3,i,4D SEDAN I,...,11597.0,12409.0,NaN,NaN,21973,33619,FL,7100.0,0,1113
1,2,0,12/7/2009,ADESA,2004,5,DODGE,1500 RAM PICKUP 2WD,ST,QUAD CAB 4.7L SLT,...,11374.0,12791.0,NaN,NaN,19638,33619,FL,7600.0,0,1053


In [61]:
df.isna().sum()

RefId                                    0
IsBadBuy                                 0
PurchDate                                0
Auction                                  0
VehYear                                  0
VehicleAge                               0
Make                                     0
Model                                    0
Trim                                  2360
SubModel                                 8
Color                                    8
Transmission                             9
WheelTypeID                           3169
WheelType                             3174
VehOdo                                   0
Nationality                              5
Size                                     5
TopThreeAmericanName                     5
MMRAcquisitionAuctionAveragePrice       18
MMRAcquisitionAuctionCleanPrice         18
MMRAcquisitionRetailAveragePrice        18
MMRAcquisitonRetailCleanPrice           18
MMRCurrentAuctionAveragePrice          315
MMRCurrentA

Fill missing cat features with word "missing"

In [62]:
cat_features_with_missing = ['Trim', 'SubModel', 'Color', 'Size', 'Transmission', 'WheelType', 'Nationality', 'Size', 'TopThreeAmericanName', 'AUCGUART', 'PRIMEUNIT']
for col in cat_features_with_missing:
    df[col] = df[col].fillna("missing")

Fill missing float features with median

In [63]:
numeric_features = ['WheelTypeID', 'MMRAcquisitionAuctionAveragePrice', 'MMRAcquisitionAuctionCleanPrice', 'MMRAcquisitionRetailAveragePrice', 'MMRAcquisitonRetailCleanPrice', 'MMRCurrentAuctionAveragePrice', 'MMRCurrentAuctionCleanPrice', 'MMRCurrentRetailAveragePrice', 'MMRCurrentRetailCleanPrice']
for col in numeric_features:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

In [64]:
df.isna().sum()

RefId                                0
IsBadBuy                             0
PurchDate                            0
Auction                              0
VehYear                              0
VehicleAge                           0
Make                                 0
Model                                0
Trim                                 0
SubModel                             0
Color                                0
Transmission                         0
WheelTypeID                          0
WheelType                            0
VehOdo                               0
Nationality                          0
Size                                 0
TopThreeAmericanName                 0
MMRAcquisitionAuctionAveragePrice    0
MMRAcquisitionAuctionCleanPrice      0
MMRAcquisitionRetailAveragePrice     0
MMRAcquisitonRetailCleanPrice        0
MMRCurrentAuctionAveragePrice        0
MMRCurrentAuctionCleanPrice          0
MMRCurrentRetailAveragePrice         0
MMRCurrentRetailCleanPric

__Task_2__ - Splitting data ratio 0.33/0.33/0.33 by 'PurchDate'

In [65]:
date_threshold_1 = pd.to_datetime("2009-09-15")
date_threshold_2 = pd.to_datetime("2010-05-15")
df['PurchDate'] = pd.to_datetime(df['PurchDate'])

df_train = df[df['PurchDate'] < date_threshold_1]
df_val = df[(df['PurchDate'] >= date_threshold_1) & (df['PurchDate'] < date_threshold_2)]
df_test = df[df['PurchDate'] >= date_threshold_2]

print(f"df_train: {df_train.shape}, percentile_shape : {int((df_train.shape[0]/df.shape[0])*100)}%")
print(f"df_val: {df_val.shape}, percentile_shape : {int(df_val.shape[0]/df.shape[0]*100)}%")
print(f"df_test: {df_test.shape}, percentile_shape : {int(df_test.shape[0]/df.shape[0]*100)}%")

df_train: (24232, 34), percentile_shape : 33%
df_val: (24486, 34), percentile_shape : 33%
df_test: (24265, 34), percentile_shape : 33%


In [66]:
print('Checking:')
if max(df_train['PurchDate']) < max(df_val['PurchDate']) < min(df_test['PurchDate']):
    print("train < val < test")
else: 
    print("Uvays")

df_train = df_train.drop(['PurchDate', 'RefId'], axis=1)
df_val = df_val.drop(['PurchDate', 'RefId'], axis=1)
df_test = df_test.drop(['PurchDate', 'RefId'], axis=1)

Checking:
train < val < test


__Task_3__ - Preprocessing (Cat_features + scaling)

In [67]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 24232 entries, 426 to 72828
Data columns (total 32 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   IsBadBuy                           24232 non-null  int64  
 1   Auction                            24232 non-null  object 
 2   VehYear                            24232 non-null  int64  
 3   VehicleAge                         24232 non-null  int64  
 4   Make                               24232 non-null  object 
 5   Model                              24232 non-null  object 
 6   Trim                               24232 non-null  object 
 7   SubModel                           24232 non-null  object 
 8   Color                              24232 non-null  object 
 9   Transmission                       24232 non-null  object 
 10  WheelTypeID                        24232 non-null  float64
 11  WheelType                          24232 non-null  object

In [68]:
df_cat_features = df_train.select_dtypes(include='object')
for cat in df_cat_features.columns:
    print(f"{cat}: {df[cat].nunique()}")

Auction: 3
Make: 33
Model: 1063
Trim: 135
SubModel: 864
Color: 17
Transmission: 4
WheelType: 4
Nationality: 5
Size: 13
TopThreeAmericanName: 5
PRIMEUNIT: 3
AUCGUART: 3
VNST: 37


Features for OneHotEncoder: ['Auction', 'Transmission', 'WheelType', 'Nationality', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART'] \ +14
Need more researching: ['Model', 'SubModel', 'Trim', 'Make', 'VNST', 'Size', 'Color']

In [69]:
features_for_OHE = ['Auction', 'Transmission', 'WheelType', 'Nationality', 'TopThreeAmericanName', 'PRIMEUNIT', 'AUCGUART']

ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(df_train[features_for_OHE])

X_train = pd.DataFrame(ohe.transform(df_train[features_for_OHE]), columns=ohe.get_feature_names_out(features_for_OHE), index=df_train.index)
X_val   = pd.DataFrame(ohe.transform(df_val[features_for_OHE]),   columns=ohe.get_feature_names_out(features_for_OHE), index=df_val.index)
X_test  = pd.DataFrame(ohe.transform(df_test[features_for_OHE]),  columns=ohe.get_feature_names_out(features_for_OHE), index=df_test.index)

df_train = df_train.drop(columns=features_for_OHE).join(X_train)
df_val   = df_val.drop(columns=features_for_OHE).join(X_val)
df_test  = df_test.drop(columns=features_for_OHE).join(X_test)

In [70]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

(24232, 47)
(24486, 47)
(24265, 47)


In [71]:
cat_features = df_train.select_dtypes(include='object')
print(cat_features.columns)

Index(['Make', 'Model', 'Trim', 'SubModel', 'Color', 'Size', 'VNST'], dtype='object')


['Make', 'Color', 'Size', 'VNST'] - Frequency encoding \
['Trim', 'Model', 'SubModel'] - Hashing encoding

In [72]:

freq_features = ['Make', 'Color', 'Size', 'VNST']
for col in freq_features:
    freq = df_train[col].value_counts(normalize=True)
    df_train[col] = df_train[col].map(freq)
    df_val[col] = df_val[col].map(freq).fillna(0)
    df_test[col] = df_test[col].map(freq).fillna(0)

hash_cols = ['Trim', 'Model', 'SubModel']
hash_enc = HashingEncoder(cols=hash_cols, n_components=8)
hash_enc.fit(df_train[hash_cols])

df_train_hash = hash_enc.transform(df_train[hash_cols])
df_val_hash = hash_enc.transform(df_val[hash_cols])
df_test_hash = hash_enc.transform(df_test[hash_cols])

df_train = df_train.drop(columns=hash_cols).join(df_train_hash)
df_val = df_val.drop(columns=hash_cols).join(df_val_hash)
df_test = df_test.drop(columns=hash_cols).join(df_test_hash)


In [73]:
print(f"'train {df_train.shape}' - 'val {df_val.shape}' - 'test {df_test.shape}'")
df_train.sample(5)

'train (24232, 52)' - 'val (24486, 52)' - 'test (24265, 52)'


,IsBadBuy,VehYear,VehicleAge,Make,Color,WheelTypeID,VehOdo,Size,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,...,AUCGUART_GREEN,AUCGUART_missing,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7
51436,0,2008,1,0.232915,0.140806,2.0,73252,0.121657,9216.0,10368.0,...,0.0,1.0,0,0,1,0,2,0,0,0
61585,0,2005,4,0.113899,0.140806,2.0,41473,0.375743,4243.0,5195.0,...,0.0,1.0,0,0,1,0,0,2,0,0
26326,0,2005,4,0.232915,0.173159,1.0,77493,0.122318,6485.0,7515.0,...,0.0,1.0,0,0,0,0,1,1,0,1
34972,0,2004,5,0.113899,0.140806,1.0,64303,0.028268,6704.0,7703.0,...,0.0,1.0,0,2,0,0,0,0,1,0
13921,0,2001,8,0.175388,0.101684,1.0,81460,0.101519,1961.0,2868.0,...,0.0,1.0,0,0,1,1,0,1,0,0


In [74]:
X_train = df_train.drop('IsBadBuy', axis=1)
X_val = df_val.drop('IsBadBuy', axis=1)
X_test = df_test.drop('IsBadBuy', axis=1)

y_train = df_train['IsBadBuy']
y_val = df_val['IsBadBuy']
y_test = df_test['IsBadBuy']

scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

X_train_scaled.head(3)

,VehYear,VehicleAge,Make,Color,WheelTypeID,VehOdo,Size,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,...,AUCGUART_GREEN,AUCGUART_missing,col_0,col_1,col_2,col_3,col_4,col_5,col_6,col_7
426,0.110667,-0.110689,-1.609681,-1.690524,1.052653,-1.213321,1.259723,-0.765496,-0.758456,-0.757905,...,-0.009085,0.009085,-0.852686,-0.662279,0.617200,-0.508968,1.021269,-0.576819,-0.545039,1.325292
427,0.110667,-0.110689,-0.373599,0.774110,-0.881081,-0.511006,-1.166303,0.303685,0.283015,0.304252,...,-0.009085,0.009085,-0.852686,1.158087,-0.877651,-0.508968,-0.680869,1.191139,1.468282,-0.616685
428,0.734316,-0.734315,1.091011,-0.452189,1.052653,-0.095088,-0.514273,0.365823,0.263981,0.366212,...,-0.009085,0.009085,0.839973,1.158087,-0.877651,-0.508968,1.021269,-0.576819,-0.545039,-0.616685


__Task_4__ - LogReg, GaussianNB, KNN

In [75]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict_proba(X_val_scaled)[:, 1]

nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
nb_pred = nb.predict_proba(X_val_scaled)[:, 1]

knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict_proba(X_val_scaled)[:, 1]

In [76]:
def gini_score(y_true, y_pred):
    return round(2 * roc_auc_score(y_true, y_pred) - 1, 3)

print("Logistic Regression Gini:", gini_score(y_val, lr_pred))
print("Naive Bayes Gini:", gini_score(y_val, nb_pred))
print("KNN Gini:", gini_score(y_val, knn_pred))

Logistic Regression Gini: 0.445
Naive Bayes Gini: 0.453
KNN Gini: 0.332


Naive Bayes Gini - the best model. Naive Bayes works better when feaures independent. LogReg requirement linear dependency, KNN works worse when features become more 10.

__Task_5__ - Implementation Custom AUC and comparing with AUC from Sklearn

In [77]:
def custom_auc(y_true, y_scores):
    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)
    desc_score_indices = np.argsort(y_scores)[::-1]
    y_true_sorted = y_true[desc_score_indices]
    pos_count = np.sum(y_true_sorted == 1)
    neg_count = np.sum(y_true_sorted == 0)
    if pos_count == 0 or neg_count == 0: return 0.0
    cum_neg, rank_sum = 0, 0
    for label in y_true_sorted:
        if label == 1: rank_sum += cum_neg
        else: cum_neg += 1
    auc = rank_sum / (pos_count * neg_count)
    return auc

def gini_from_auc(auc):
    return -round(2 * auc - 1, 3)

In [78]:
auc_lr_custom = custom_auc(y_val, lr_pred)
gini_lr_custom = gini_from_auc(auc_lr_custom)

auc_nb_custom = custom_auc(y_val, nb_pred)
gini_nb_custom = gini_from_auc(auc_nb_custom)

auc_knn_custom = custom_auc(y_val, knn_pred)
gini_knn_custom = gini_from_auc(auc_knn_custom)

print("Logistic Regression Gini Custom:", gini_lr_custom)
print("Naive Bayes Gini Custom:", gini_nb_custom)
print("KNN Gini Custom:", gini_knn_custom)

Logistic Regression Gini Custom: 0.445
Naive Bayes Gini Custom: 0.453
KNN Gini Custom: 0.324


__Task_6__ - Implementation of LogRreg, KNN, NaiveBayes

In [79]:
class MyLogisticRegression:
    def __init__(self, lr=0.01, n_iter=1000, reg_lambda=0.1):
        self.lr = lr
        self.n_iter = n_iter
        self.reg_lambda = reg_lambda
        self.w = None
        self.b = None

    def sigmoid(self, z):
        z = np.array(z)
        out = np.empty_like(z)
        mask = z >= 0
        out[mask] = 1 / (1 + np.exp(-z[mask]))
        exp_z = np.exp(z[~mask])
        out[~mask] = exp_z / (1 + exp_z)
        return out

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0
        for _ in range(self.n_iter):
            linear_model = np.dot(X, self.w) + self.b
            y_pred = self.sigmoid(linear_model)
            error = y_pred - y
            dw = (np.dot(X.T, error) / n_samples) + self.reg_lambda * self.w
            db = np.sum(error) / n_samples
            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        X = np.array(X)
        linear_model = np.dot(X, self.w) + self.b
        return self.sigmoid(linear_model)

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba >= 0.5).astype(int)

In [80]:
class MyKNN:
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X_train = np.array(X)
        self.y_train = np.array(y)

    def _predict_proba(self, x):
        distances = np.linalg.norm(self.X_train - x, axis=1)
        k_indices = distances.argsort()[:self.k]
        k_neighbor_labels = self.y_train[k_indices]
        proba_1 = np.mean(k_neighbor_labels)
        return [1 - proba_1, proba_1]

    def predict_proba(self, X):
        X = np.array(X)
        return np.array([self._predict_proba(x) for x in X])

    def predict(self, X):
        probs = self.predict_proba(X)
        return (probs[:, 1] >= 0.5).astype(int)

In [81]:
class MyGaussianNB:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.mean = {}
        self.var = {}
        self.priors = {}

        for c in self.classes:
            X_c = X[y == c]
            self.mean[c] = X_c.mean(axis=0)
            self.var[c] = X_c.var(axis=0) + 1e-9
            self.priors[c] = X_c.shape[0] / X.shape[0]

    def predict_proba(self, X):
        return np.array([self._posterior(x) for x in X.values])

    def predict(self, X):
        probs = self.predict_proba(X)
        return np.argmax(probs, axis=1)

    def _posterior(self, x):
        x = np.asarray(x, dtype=float)
        posteriors = []
        for c in self.classes:
            prior = np.log(self.priors[c])
            class_conditional = -0.5 * np.sum(np.log(2 * np.pi * self.var[c]))
            class_conditional -= 0.5 * np.sum(((x - self.mean[c]) ** 2) / self.var[c])
            posteriors.append(prior + class_conditional)
        probs = np.exp(posteriors - np.max(posteriors))
        probs /= np.sum(probs)
        return probs

In [82]:
my_logreg = MyLogisticRegression(lr=0.01, n_iter=30000)
my_logreg.fit(X_train_scaled, y_train)
my_logreg_pred = my_logreg.predict_proba(X_val_scaled)

my_bayes = MyGaussianNB()
my_bayes.fit(X_train_scaled, y_train)
my_bayes_pred = my_bayes.predict_proba(X_val_scaled)[:, 1]

my_knn = MyKNN()
my_knn.fit(X_train_scaled, y_train)
my_knn_pred = my_knn.predict_proba(X_val_scaled)[:, 1]

In [83]:
print("My Logistic Regression Gini:", gini_score(y_val, my_logreg_pred))
print("My Naive Bayes Gini:", gini_score(y_val, my_bayes_pred))
print("My KNN Gini:", gini_score(y_val, my_knn_pred))

My Logistic Regression Gini: 0.465
My Naive Bayes Gini: 0.453
My KNN Gini: 0.332


__Task_7__ - Gini with Non-linear features # Подумать над другими пропорциями

In [84]:
X_train['price_diff_1'] = X_train['MMRAcquisitionAuctionCleanPrice'] - X_train['MMRAcquisitionAuctionAveragePrice']
X_train['price_diff_2'] = X_train['MMRCurrentRetailCleanPrice'] - X_train['MMRAcquisitionAuctionCleanPrice']
X_train['clean_to_avg_ratio'] = X_train['MMRAcquisitionAuctionCleanPrice'] / (X_train['MMRAcquisitionAuctionAveragePrice'] + 1e-6)
X_train['retail_vs_cost'] = X_train['MMRCurrentRetailCleanPrice'] / (X_train['VehBCost'] + 1e-6)
X_train['retail_to_clean_ratio'] = X_train['MMRCurrentRetailCleanPrice'] / (X_train['MMRAcquisitionAuctionCleanPrice'] + 1e-6)
X_train['odo_to_cost_ratio'] = X_train['VehOdo'] / (X_train['VehBCost'] + 1e-6)
X_train['mean_cost_by_make'] = X_train['Make'].map(X_train.groupby('Make')['VehBCost'].mean())
X_train['mean_age_by_color'] = X_train['Color'].map(X_train.groupby('Color')['VehicleAge'].mean())
X_train['mean_warranty_by_size'] = X_train['Size'].map(X_train.groupby('Size')['WarrantyCost'].mean())
X_train['mean_odo_by_make'] = X_train['Make'].map(X_train.groupby('Make')['VehOdo'].mean())

X_val['price_diff_1'] = X_val['MMRAcquisitionAuctionCleanPrice'] - X_val['MMRAcquisitionAuctionAveragePrice']
X_val['price_diff_2'] = X_val['MMRCurrentRetailCleanPrice'] - X_val['MMRAcquisitionAuctionCleanPrice']
X_val['clean_to_avg_ratio'] = X_val['MMRAcquisitionAuctionCleanPrice'] / (X_val['MMRAcquisitionAuctionAveragePrice'] + 1e-6)
X_val['retail_vs_cost'] = X_val['MMRCurrentRetailCleanPrice'] / (X_val['VehBCost'] + 1e-6)
X_val['retail_to_clean_ratio'] = X_val['MMRCurrentRetailCleanPrice'] / (X_val['MMRAcquisitionAuctionCleanPrice'] + 1e-6)
X_val['odo_to_cost_ratio'] = X_val['VehOdo'] / (X_val['VehBCost'] + 1e-6)
X_val['mean_cost_by_make'] = X_val['Make'].map(X_val.groupby('Make')['VehBCost'].mean())
X_val['mean_age_by_color'] = X_val['Color'].map(X_val.groupby('Color')['VehicleAge'].mean())
X_val['mean_warranty_by_size'] = X_val['Size'].map(X_val.groupby('Size')['WarrantyCost'].mean())
X_val['mean_odo_by_make'] = X_val['Make'].map(X_val.groupby('Make')['VehOdo'].mean())

X_test['price_diff_1'] = X_test['MMRAcquisitionAuctionCleanPrice'] - X_test['MMRAcquisitionAuctionAveragePrice']
X_test['price_diff_2'] = X_test['MMRCurrentRetailCleanPrice'] - X_test['MMRAcquisitionAuctionCleanPrice']
X_test['clean_to_avg_ratio'] = X_test['MMRAcquisitionAuctionCleanPrice'] / (X_test['MMRAcquisitionAuctionAveragePrice'] + 1e-6)
X_test['retail_vs_cost'] = X_test['MMRCurrentRetailCleanPrice'] / (X_test['VehBCost'] + 1e-6)
X_test['retail_to_clean_ratio'] = X_test['MMRCurrentRetailCleanPrice'] / (X_test['MMRAcquisitionAuctionCleanPrice'] + 1e-6)
X_test['odo_to_cost_ratio'] = X_test['VehOdo'] / (X_test['VehBCost'] + 1e-6)
X_test['mean_cost_by_make'] = X_test['Make'].map(X_test.groupby('Make')['VehBCost'].mean())
X_test['mean_age_by_color'] = X_test['Color'].map(X_test.groupby('Color')['VehicleAge'].mean())
X_test['mean_warranty_by_size'] = X_test['Size'].map(X_test.groupby('Size')['WarrantyCost'].mean())
X_test['mean_odo_by_make'] = X_test['Make'].map(X_test.groupby('Make')['VehOdo'].mean())

In [85]:
scaler = StandardScaler()
scaler.fit(X_train)

X_train_scaled_new = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_scaled_new   = pd.DataFrame(scaler.transform(X_val),   columns=X_val.columns,   index=X_val.index)
X_test_scaled_new  = pd.DataFrame(scaler.transform(X_test),  columns=X_test.columns,  index=X_test.index)

In [86]:
print(X_train_scaled_new.shape)
X_train_scaled_new.head(3)

(24232, 61)


,VehYear,VehicleAge,Make,Color,WheelTypeID,VehOdo,Size,MMRAcquisitionAuctionAveragePrice,MMRAcquisitionAuctionCleanPrice,MMRAcquisitionRetailAveragePrice,...,price_diff_1,price_diff_2,clean_to_avg_ratio,retail_vs_cost,retail_to_clean_ratio,odo_to_cost_ratio,mean_cost_by_make,mean_age_by_color,mean_warranty_by_size,mean_odo_by_make
426,0.110667,-0.110689,-1.609681,-1.690524,1.052653,-1.213321,1.259723,-0.765496,-0.758456,-0.757905,...,-0.459424,3.430811,-0.073725,0.020106,-0.01051,-0.008710,-1.172734,1.220323,-0.720252,1.409518
427,0.110667,-0.110689,-0.373599,0.774110,-0.881081,-0.511006,-1.166303,0.303685,0.283015,0.304252,...,0.069299,-0.078849,-0.073726,-0.012288,-0.01051,-0.012561,0.060483,0.080438,0.076285,-0.875891
428,0.734316,-0.734315,1.091011,-0.452189,1.052653,-0.095088,-0.514273,0.365823,0.263981,0.366212,...,-0.402694,0.014893,-0.073726,-0.005664,-0.01051,-0.009431,0.210421,-0.248555,1.413274,0.364605


In [87]:
lr = LogisticRegression(max_iter=5000)
lr.fit(X_train_scaled_new, y_train)
lr_pred = lr.predict_proba(X_val_scaled_new)[:, 1]

nb = GaussianNB()
nb.fit(X_train_scaled_new, y_train)
nb_pred = nb.predict_proba(X_val_scaled_new)[:, 1]

knn = KNeighborsClassifier()
knn.fit(X_train_scaled_new, y_train)
knn_pred = knn.predict_proba(X_val_scaled_new)[:, 1]

In [88]:
print("Logistic Regression Gini:", gini_score(y_val, lr_pred))
print("Naive Bayes Gini:", gini_score(y_val, nb_pred))
print("KNN Gini:", gini_score(y_val, knn_pred))

Logistic Regression Gini: 0.459
Naive Bayes Gini: 0.441
KNN Gini: 0.328


__Task_8__ - Manual vs Lasso feature selection method using logreg

In [89]:
model = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
model.fit(X_train_scaled, y_train)
gini_manual = 2 * roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1]) - 1

In [90]:
model_l1 = LogisticRegression(penalty='l1', solver='liblinear', C=1.0)
model_l1.fit(X_train_scaled, y_train)
gini_l1 = 2 * roc_auc_score(y_val, model_l1.predict_proba(X_val_scaled)[:, 1]) - 1

In [91]:
coefs_before = pd.Series(abs(model.coef_[0]), index=X_train_scaled.columns).sort_values()
print(coefs_before)

IsOnlineSale                         0.000000
col_3                                0.007975
col_5                                0.009590
Auction_MANHEIM                      0.010798
col_0                                0.012188
Nationality_AMERICAN                 0.012435
TopThreeAmericanName_OTHER           0.012435
TopThreeAmericanName_CHRYSLER        0.012507
Size                                 0.012760
Make                                 0.015895
col_7                                0.016908
Transmission_MANUAL                  0.017083
Nationality_TOP LINE ASIAN           0.020051
Nationality_OTHER ASIAN              0.022442
col_1                                0.023271
Transmission_AUTO                    0.024038
Color                                0.026446
WheelType_Special                    0.027565
col_6                                0.027641
col_2                                0.030046
Auction_ADESA                        0.030918
col_4                             

In [92]:
print(f"Gini (without feature selection): {gini_manual:.4f}")
print(f"Gini (L1 regularization): {gini_l1:.4f}")

Gini (without feature selection): 0.4136
Gini (L1 regularization): 0.4610


In [93]:
threshold = 0.01
important_features = coefs_before[coefs_before.abs() > threshold].index
X_train_filtered = X_train_scaled[important_features]
X_val_filtered = X_val_scaled[important_features]

model.fit(X_train_filtered, y_train)
gini_filtered = 2 * roc_auc_score(y_val, model.predict_proba(X_val_filtered)[:, 1]) - 1

In [94]:
coefs_raw = model.coef_[0]
mask_nonzero = coefs_raw != 0
selected_columns = X_train_filtered.columns[mask_nonzero]

coefs_after = pd.Series(abs(coefs_raw[mask_nonzero]), index=selected_columns).sort_values()
print(coefs_after)

col_0                                0.009749
Auction_MANHEIM                      0.010834
Size                                 0.011311
TopThreeAmericanName_OTHER           0.011407
Nationality_AMERICAN                 0.011407
TopThreeAmericanName_CHRYSLER        0.014543
Make                                 0.017055
Transmission_MANUAL                  0.017660
col_7                                0.018774
Nationality_TOP LINE ASIAN           0.021096
Nationality_OTHER ASIAN              0.022018
col_1                                0.022275
Transmission_AUTO                    0.024600
Color                                0.026128
WheelType_Special                    0.027374
col_2                                0.028591
col_6                                0.031162
Auction_ADESA                        0.031187
col_4                                0.033941
Nationality_OTHER                    0.036261
VNST                                 0.037897
PRIMEUNIT_YES                     

In [95]:
print(f"Gini (with feature selection): {gini_filtered:.4f}")
print(f"Gini (L1 regularization): {gini_l1:.4f}")

Gini (with feature selection): 0.4108
Gini (L1 regularization): 0.4610


Ручной способ feature selection через trashold плох, поскольку он "слепой", не смотря на то что мы убираем околонулевые признаки, они на самом деле могут зависеть от других.

__Task_9__ - SVM and explanation about equality of MLE and NLL

In [96]:
svc = SVC(kernel='rbf', probability=True, random_state=42)
svc.fit(X_train_scaled, y_train)

y_val_proba = svc.predict_proba(X_val_scaled)[:, 1]
auc = roc_auc_score(y_val, y_val_proba)
gini = 2 * auc - 1

print(f"Gini Score: {gini:.3f}")

Gini Score: 0.347


L(θ)=i=1∏n​piyi​​(1−pi​)1−yi​ \
MLE (максимум правдоподобия) - это максимизация L(θ) \
logL(θ)=i=1∑n​[yi​logpi​+(1−yi​)log(1−pi​)] 

NLL (негативный логарифм правдоподобия) - отрицательный логарифм правдоподобия \
NLL(θ)=−i=1∑n​[yi​logpi​+(1−yi​)log(1−pi​)] 

MLE = minimize(NLL)

__Task_10__ - Hyperparameters

In [97]:
logreg = LogisticRegression(random_state=42)
logreg.fit(X_train_scaled, y_train)

y_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
auc = roc_auc_score(y_val, y_val_proba)
gini = 2 * auc - 1

print(f"Gini for our best model: {gini:.3f}")

Gini for our best model: 0.445


In [98]:
param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'saga']
}

logreg = LogisticRegression(random_state=42, class_weight = 'balanced', max_iter=5000)
grid = GridSearchCV(logreg, param_grid, scoring='roc_auc', cv=5, n_jobs=-1)
grid.fit(X_train_scaled, y_train)

best_model = grid.best_estimator_
y_val_proba = best_model.predict_proba(X_val_scaled)[:, 1]
auc = roc_auc_score(y_val, y_val_proba)
gini = 2 * auc - 1

print(f"Best Gini: {gini:.3f}")
print("Best Params:", grid.best_params_)

Best Gini: 0.479
Best Params: {'C': 0.01, 'penalty': 'l1', 'solver': 'liblinear'}


__Task_11__ - comparing Gini on 3 datasets

In [99]:
y_train_proba = best_model.predict_proba(X_train_scaled)[:, 1]
y_val_proba = best_model.predict_proba(X_val_scaled)[:, 1]
y_test_proba = best_model.predict_proba(X_test_scaled)[:, 1]

auc_train = roc_auc_score(y_train, y_train_proba)
auc_val = roc_auc_score(y_val, y_val_proba)
auc_test = roc_auc_score(y_test, y_test_proba)

gini_train = 2 * auc_train - 1
gini_val = 2 * auc_val - 1
gini_test = 2 * auc_test - 1

print(f"Train Gini: {gini_train:.3f}")
print(f"Validation Gini: {gini_val:.3f}")
print(f"Test Gini: {gini_test:.3f}")

Train Gini: 0.506
Validation Gini: 0.479
Test Gini: 0.351


Наблюдается переобучение, поскольку на тестовом датасете видим явное падение индекса Джини, причем параметры тут не влияют на переобучение, возможная причина дисбаланс классов, нужно проводить доп. исследование.

__Task_12__ - Implementation of functions precision, recall, f1, roc_auc and comparing with sklearn.

In [100]:
def custom_precision_recall_f1(y_true, y_pred_labels):
    tp = np.sum((y_true == 1) & (y_pred_labels == 1))
    fp = np.sum((y_true == 0) & (y_pred_labels == 1))
    fn = np.sum((y_true == 1) & (y_pred_labels == 0))

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    return precision, recall, f1

In [101]:
def custom_auc_pr(y_true, y_scores):
    desc_score_indices = np.argsort(-y_scores)
    y_true = np.array(y_true)[desc_score_indices]
    y_scores = np.array(y_scores)[desc_score_indices]

    precision, recall, tp, fp = [], [], 0, 0
    fn = np.sum(y_true == 1)

    for i in range(len(y_scores)):
        if y_true[i] == 1:
            tp += 1
            fn -= 1
        else:
            fp += 1

        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        r = tp / (tp + fn + 1e-10)

        precision.append(p)
        recall.append(r)

    auc_pr = 0.0
    for i in range(1, len(recall)):
        auc_pr += (recall[i] - recall[i - 1]) * precision[i]

    return auc_pr

In [102]:
y_test_proba = best_model.predict_proba(X_test_scaled)[:, 1]
y_test_pred = best_model.predict(X_test_scaled)

precision, recall, f1 = custom_precision_recall_f1(y_test, y_test_pred)
custom_ap = custom_auc_pr(y_test, y_test_proba)
sklearn_ap = average_precision_score(y_test, y_test_proba)

print("Custom Precision:", round(precision, 3))
print("Custom Recall:", round(recall, 3))
print("Custom F1 Score:", round(f1, 3))
print("Custom AUC PR:", round(custom_ap, 3))
print("Sklearn AUC PR:", round(sklearn_ap, 3))

Custom Precision: 0.211
Custom Recall: 0.55
Custom F1 Score: 0.305
Custom AUC PR: 0.194
Sklearn AUC PR: 0.194


__Task_13__ - Explanation of "lemon" task

Для задачи обнаружения "lemon" машин я предпочёл бы использовать метрику precision, в сочетании с жестким порогом вероятности (например, 0.75), чтобы быть уверенным в каждом положительном предсказании.

Такой подход снижает риск ошибочной классификации хорошей машины как плохой (false positive), что критично в реальной бизнес-задаче. Порог в 75% выбран не случайно - он отражает необходимость высокой уверенности в редком и дорогостоящем классе, особенно при сильном дисбалансе (около 11-13% "lemon"-машин).

Это решение можно интерпретировать как голосование (например, среди людей или моделей), где "машина считается плохой", только если «за» проголосовало 75% и более - повышая надёжность.